# Football Player Market Value Prediction & Scouting Dashboard
## Notebook 1 — Data Collection

**Description:** Collects and verifies the raw data for this project — scraped FBref player stats across the Big 5 European leagues (Premier League, La Liga, Bundesliga, Serie A, Ligue 1) for 6 seasons (2020-21 to 2025-26), the Kaggle FBref datasets, Transfermarkt market valuations and player profiles, the FBref-Transfermarkt player mapping, and injury records. Cleaning and merging happens in notebook 2.

### Data Sources
- **FBref** — scraped directly, 7 stat categories per season
- **Kaggle FBref datasets** — pre-scraped, richer stat coverage (2022-23 to 2025-26), used for the primary models
- **Transfermarkt** — market valuations and player profiles (downloaded separately)
- **Player mapping** — FBref to Transfermarkt player ID mapping (downloaded separately)
- **Injury records** — downloaded separately


In [3]:
import soccerdata as sd
import pandas as pd
from io import BytesIO
import time
import os
import re
from pathlib import Path

RAW_FBREF = Path("../data/raw/fbref/")
RAW_KAGGLE = Path("../data/raw/fbref_kaggle/")
RAW_TM = Path("../data/raw/transfermarkt/football-data/")
RAW_INJURIES = Path("../data/raw/injuries/")
EXTERNAL = Path("../data/external/")
PROCESSED = Path("../data/processed/")

for p in [RAW_FBREF, RAW_KAGGLE, RAW_TM, RAW_INJURIES, EXTERNAL, PROCESSED]:
    p.mkdir(parents=True, exist_ok=True)

print("Paths ready.")

Paths ready.


## Step 1 — Scrape All Stat Types Across All 6 Seasons
Pulls all 10 stat categories across the 6 available seasons (2020-21 to 2025-26), retrying automatically if a table comes back malformed.

In [ ]:
fbref = sd.FBref("Big 5 European Leagues Combined", ["2425"])

STAT_TYPES = ["standard", "shooting", "misc", "passing", "defense", "possession", "gca", "passing_types", "keeper", "playing_time"]

SEASONS = {
    "2021": "2020-2021",
    "2122": "2021-2022",
    "2223": "2022-2023",
    "2324": "2023-2024",
    "2425": "2024-2025",
    "2526": "",
}

MIN_EXPECTED_COLS = {
    "keeper": 10,  # smaller table, fewer columns than the rest
}
DEFAULT_MIN_COLS = 15

def scrape_fbref_big5(stat_type, season_code, season_url_part):
    if season_code == "2526":
        url = f"https://fbref.com/en/comps/Big5/{stat_type}/players/Big-5-European-Leagues-Stats"
    else:
        url = f"https://fbref.com/en/comps/Big5/{season_url_part}/{stat_type}/players/{season_url_part}-Big-5-European-Leagues-Stats"

    response = fbref.get(url)
    content = response.read()
    tables = pd.read_html(BytesIO(content))
    df = tables[-1]
    df.columns = ['_'.join(col).strip('_') for col in df.columns]
    df['season'] = season_code
    return df


def scrape_with_retry(stat_type, season_code, season_url_part, max_attempts=3, wait=5):
    min_cols = MIN_EXPECTED_COLS.get(stat_type, DEFAULT_MIN_COLS)
    for attempt in range(1, max_attempts + 1):
        df = scrape_fbref_big5(stat_type, season_code, season_url_part)
        if df.shape[1] >= min_cols:
            return df
        print(f"    attempt {attempt}: got shape {df.shape}, retrying")
        time.sleep(wait)
    print(f"    still small after {max_attempts} attempts, saving anyway")
    return df


print("scraping 6 seasons x 10 stat types\n")

for season_code, season_url in SEASONS.items():
    print(f"=== season {season_code} ===")
    for stat in STAT_TYPES:
        print(f"  pulling {stat}...")
        df = scrape_with_retry(stat, season_code, season_url)
        filename = f"fbref_{stat}_{season_code}.csv"
        df.to_csv(RAW_FBREF / filename, index=False)
        print(f"  saved {filename} — shape {df.shape}")
        time.sleep(4)
    print()

print("done scraping")

[07/23/26 09:43:56] INFO     Saving cached data to /Users/ashish/soccerdata/data/FBref               _common.py:250

scraping 6 seasons x 10 stat types

=== season 2021 ===
  pulling standard...


KeyboardInterrupt: 

## Step 2 — Combine Individual Season Files Into All-Seasons Files
Combines each stat type's per-season files into one all-seasons file. standard, keeper and playing_time are excluded — confirmed malformed (shape (5, 6) every season) in an earlier run. The other 7 combine cleanly into shape (17862, N) files.

In [5]:
WORKING_STAT_TYPES = ["shooting", "misc", "passing", "defense", "possession", "gca", "passing_types"]

def clean_identity_columns(df):
    rename = {}
    for col in df.columns:
        m = re.match(r'^Unnamed: \d+_level_0_(.+)$', col)
        if m:
            rename[col] = m.group(1)
        elif 'Matches' in col:
            rename[col] = 'Matches'
    return df.rename(columns=rename)


print("combining season files\n")

combined_frames = {}
for stat in WORKING_STAT_TYPES:
    dfs = []
    for season_code in SEASONS.keys():
        filepath = RAW_FBREF / f"fbref_{stat}_{season_code}.csv"
        df = pd.read_csv(filepath)
        df = clean_identity_columns(df)
        dfs.append(df)
    combined = pd.concat(dfs, ignore_index=True)
    combined = combined.drop(columns=[c for c in ['Rk', 'Matches'] if c in combined.columns])

    out_path = RAW_FBREF / f"fbref_{stat}_all_seasons.csv"
    combined.to_csv(out_path, index=False)
    combined_frames[stat] = combined
    print(f"  fbref_{stat}_all_seasons.csv — shape {combined.shape}")

print("\ncombined files done, columns cleaned")

combining season files

  fbref_shooting_all_seasons.csv — shape (17862, 19)
  fbref_misc_all_seasons.csv — shape (17862, 21)
  fbref_passing_all_seasons.csv — shape (17862, 29)
  fbref_defense_all_seasons.csv — shape (17862, 25)
  fbref_possession_all_seasons.csv — shape (17862, 29)
  fbref_gca_all_seasons.csv — shape (17862, 25)
  fbref_passing_types_all_seasons.csv — shape (17862, 24)

combined files done, columns cleaned


## Step 3 — Audit Scraped Column Usability
Checks how much real data survived in each of the 7 categories, and whether any of the sparse ones share columns with misc.

In [6]:
ID_COLS = {'Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', 'season', '90s', 'Minutes_90s'}

def audit_columns(df):
    rows = []
    for col in df.columns:
        if col in ID_COLS:
            continue
        s = pd.to_numeric(df[col], errors='coerce')
        rows.append({
            'column': col,
            'pct_non_null': round(s.notna().mean() * 100, 1),
            'pct_non_null_non_zero': round(((s.notna()) & (s != 0)).mean() * 100, 1),
        })
    return pd.DataFrame(rows).sort_values('pct_non_null')


print("=== column audit — scraped fbref data ===\n")

audit_summary = []
for stat, df in combined_frames.items():
    audit = audit_columns(df)
    usable = (audit['pct_non_null'] > 5).sum()
    total = len(audit)
    print(f"{stat}: {usable}/{total} columns have >5% non-null data")
    audit_summary.append({'stat_type': stat, 'total_stat_cols': total, 'usable_cols': usable, 'shape': df.shape})

audit_summary_df = pd.DataFrame(audit_summary)
print("\nsummary:")
print(audit_summary_df.to_string(index=False))

=== column audit — scraped fbref data ===

shooting: 10/10 columns have >5% non-null data
misc: 10/12 columns have >5% non-null data
passing: 1/20 columns have >5% non-null data
defense: 2/16 columns have >5% non-null data
possession: 0/20 columns have >5% non-null data
gca: 0/16 columns have >5% non-null data
passing_types: 1/15 columns have >5% non-null data

summary:
    stat_type  total_stat_cols  usable_cols       shape
     shooting               10           10 (17862, 19)
         misc               12           10 (17862, 21)
      passing               20            1 (17862, 29)
      defense               16            2 (17862, 25)
   possession               20            0 (17862, 29)
          gca               16            0 (17862, 25)
passing_types               15            1 (17862, 24)


In [7]:
print("=== which columns actually survived in the mostly-broken files ===\n")

for stat in ['passing', 'defense', 'possession', 'gca', 'passing_types']:
    audit = audit_columns(combined_frames[stat])
    survivors = audit[audit['pct_non_null'] > 5]
    print(f"{stat}:")
    if len(survivors) == 0:
        print("  none")
    else:
        print(survivors.to_string(index=False))
    print()

=== which columns actually survived in the mostly-broken files ===

passing:
column  pct_non_null  pct_non_null_non_zero
   Ast          96.2                   44.2

defense:
      column  pct_non_null  pct_non_null_non_zero
Tackles_TklW          96.2                   79.4
         Int          96.2                   76.7

possession:
  none

gca:
  none

passing_types:
        column  pct_non_null  pct_non_null_non_zero
Pass Types_Crs          96.2                   74.1



In [8]:
print("checking overlap between misc and the defense/passing survivors\n")
print("misc columns:", combined_frames['misc'].columns.tolist())

checking overlap between misc and the defense/passing survivors

misc columns: ['Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', '90s', 'Performance_CrdY', 'Performance_CrdR', 'Performance_2CrdY', 'Performance_Fls', 'Performance_Fld', 'Performance_Off', 'Performance_Crs', 'Performance_Int', 'Performance_TklW', 'Performance_PKwon', 'Performance_PKcon', 'Performance_OG', 'season']


### Step 3 (contd.) - Findings
Of the 10 categories, only 7 return anything via a plain scrape — standard, keeper and playing_time are excluded (Step 2). Of those 7, only shooting (10/10) and misc (10/12) have most columns usable; the rest are 90%+ null, with just Ast (passing) and Pass Types_Crs (passing_types) surviving. Defense's two survivors (Tkl, Int) look like duplicates of stats already in misc. Usable total: shooting, misc, Ast, Pass Types_Crs — everything else scraped is dropped. This scopes the 6-season data to EDA only; modelling uses the Kaggle datasets instead.

## Step 4 — Load & Check Kaggle Datasets
Reads every CSV in data/raw/fbref_kaggle/ automatically, so it also picks up the _light versions. Needs to be downloaded into that folder first.

In [9]:
print("=== kaggle fbref datasets ===\n")

kaggle_files = sorted(RAW_KAGGLE.glob("*.csv"))
kaggle_summary = []
if not kaggle_files:
    print(f"no files found in {RAW_KAGGLE} — download the kaggle datasets there first")
else:
    for f in kaggle_files:
        df = pd.read_csv(f, low_memory=False)
        has_xg = any('xG' in c for c in df.columns)
        print(f"{f.name}")
        print(f"  shape: {df.shape}")
        print(f"  has xG columns: {has_xg}")
        print()
        kaggle_summary.append({'file': f.name, 'rows': df.shape[0], 'cols': df.shape[1], 'has_xg': has_xg})

    print(pd.DataFrame(kaggle_summary).to_string(index=False))

=== kaggle fbref datasets ===

players_data-2024_2025.csv
  shape: (2854, 267)
  has xG columns: True

players_data-2025_2026.csv
  shape: (2839, 102)
  has xG columns: False

players_data_light-2024_2025.csv
  shape: (2854, 165)
  has xG columns: True

players_data_light-2025_2026.csv
  shape: (2839, 53)
  has xG columns: False

                            file  rows  cols  has_xg
      players_data-2024_2025.csv  2854   267    True
      players_data-2025_2026.csv  2839   102   False
players_data_light-2024_2025.csv  2854   165    True
players_data_light-2025_2026.csv  2839    53   False


## Step 5 — Transfermarkt
Loads and checks shapes for the Transfermarkt player and valuation files.

In [12]:
RAW_TM = Path("../data/raw/transfermarkt/")

print("=== transfermarkt ===\n")

players = pd.read_csv(RAW_TM / "players.csv")
valuations = pd.read_csv(RAW_TM / "player_valuations.csv")

print(f"players.csv: {players.shape}")
print(f"player_valuations.csv: {valuations.shape}")

=== transfermarkt ===

players.csv: (48380, 26)
player_valuations.csv: (656301, 6)



## Step 6 — Check for Git LFS Pointer Files
Scans every raw/external CSV for unresolved Git LFS pointers — player_performances.csv was already known to be one, so the rest are checked for the same issue.

In [13]:
def is_lfs_pointer(filepath, max_check_bytes=200):
    try:
        with open(filepath, 'rb') as f:
            head = f.read(max_check_bytes)
        return head.startswith(b'version https://git-lfs')
    except Exception:
        return False

print("scanning raw/external csvs for lfs pointers\n")

lfs_pointers = []
for d in [RAW_FBREF, RAW_KAGGLE, RAW_TM, RAW_INJURIES, EXTERNAL]:
    for f in d.rglob("*.csv"):
        if is_lfs_pointer(f):
            lfs_pointers.append(str(f))

if lfs_pointers:
    print(f"warning — {len(lfs_pointers)} file(s) are unresolved lfs pointers:")
    for f in lfs_pointers:
        print(f"  {f}")
    print("\nrun git lfs pull in the repo to fetch the actual content")
else:
    print("no lfs pointer files found")

scanning raw/external csvs for lfs pointers

warning — 1 file(s) are unresolved lfs pointers:
  ../data/raw/injuries/player_performances/player_performances.csv

run git lfs pull in the repo to fetch the actual content


## Step 7 — Final Inventory
Verifies everything is in place and ready for notebook 2. No pre-built FBref-Transfermarkt mapping file exists this time round. Notebook 2 will need to build one via fuzzy name matching against players.csv instead of starting from one.

In [14]:
print("=" * 90)
print("final data inventory — ready for notebook 2")
print("=" * 90)

inventory = []
for stat, df in combined_frames.items():
    inventory.append({'source': 'fbref_scraped', 'file': f'fbref_{stat}_all_seasons.csv', 'rows': df.shape[0], 'cols': df.shape[1]})
for entry in kaggle_summary:
    inventory.append({'source': 'kaggle', 'file': entry['file'], 'rows': entry['rows'], 'cols': entry['cols']})
inventory.append({'source': 'transfermarkt', 'file': 'players.csv', 'rows': players.shape[0], 'cols': players.shape[1]})
inventory.append({'source': 'transfermarkt', 'file': 'player_valuations.csv', 'rows': valuations.shape[0], 'cols': valuations.shape[1]})

print(pd.DataFrame(inventory).to_string(index=False))

print("\nNOTE: fbref_to_tm_mapping.csv is missing. Notebook 2 needs to build the FBref <-> Transfermarkt")
print("player mapping from scratch via fuzzy name matching against players.csv, rather than starting")
print("from a pre-built mapping file like last time.")

print("\ndata collection complete, on to notebook 2")

final data inventory — ready for notebook 2
       source                                file   rows  cols
fbref_scraped      fbref_shooting_all_seasons.csv  17862    19
fbref_scraped          fbref_misc_all_seasons.csv  17862    21
fbref_scraped       fbref_passing_all_seasons.csv  17862    29
fbref_scraped       fbref_defense_all_seasons.csv  17862    25
fbref_scraped    fbref_possession_all_seasons.csv  17862    29
fbref_scraped           fbref_gca_all_seasons.csv  17862    25
fbref_scraped fbref_passing_types_all_seasons.csv  17862    24
       kaggle          players_data-2024_2025.csv   2854   267
       kaggle          players_data-2025_2026.csv   2839   102
       kaggle    players_data_light-2024_2025.csv   2854   165
       kaggle    players_data_light-2025_2026.csv   2839    53
transfermarkt                         players.csv  48380    26
transfermarkt               player_valuations.csv 656301     6

NOTE: fbref_to_tm_mapping.csv is missing. Notebook 2 needs to build the F